# 02 · LeNet, VGG y Batch Normalization

Entrenaremos cuatro modelos: LeNet, LeNet+BN, VGG-11 y VGG-11+BN.
Luego probaremos una tasa de aprendizaje más alta. Esto cubre las tareas 1 y 2.

## Preparación

Sube la carpeta `PC3_Deep_Learning` a **Mi unidad** de Google Drive, incluyendo
`data/raw/` con las imágenes descargadas. No hace falta subir `.venv`.
Usaremos esa carpeta en los tres notebooks. Ejecútalos en orden: **01 → 02 → 03**.
Para los notebooks 02 y 03, activa una GPU en Colab.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/PC3_Deep_Learning
%pip install -q -r requirements.txt

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as MostrarImagen
from src.utils import seed_everything

seed_everything(42)

## Modelos y parámetros

LeNet tiene dos bloques de convolución y pooling, y tres capas finales.
VGG-11 usa ocho convoluciones, cinco poolings y tres capas finales.
Sus filtros se reducen a la mitad y su parte final es más pequeña para ahorrar memoria.
Ambos reciben imágenes de 64×64. Las versiones con BN la aplican después de cada convolución.

Los pares con y sin BN usan los mismos datos, pesos iniciales compartidos,
semilla y ajustes. Los ajustes están en `src/train.py`.

In [ ]:
from src.models import build_model, parameter_counts
from src.train import DEFAULT_SETTINGS, experiment_plan, run_suite
from src.utils import export_results

parametros = []
for modelo in experiment_plan(DEFAULT_SETTINGS, "scratch"):
    seed_everything(42)
    red = build_model(modelo["architecture"], modelo["batch_norm"])
    parametros.append({"modelo": modelo["name"], **parameter_counts(red)})
display(pd.DataFrame(parametros))
display(DEFAULT_SETTINGS)

## Entrenar los cuatro modelos

Ejecuta primero el notebook 01 para preparar las particiones.
Usaremos GPU, 20 épocas, batch 64 y SGD con LR 0.01.
Se guarda la mejor época según validación y después se evalúa en test.

In [ ]:
resultados = run_suite(DEFAULT_SETTINGS, "scratch", device="cuda")
display(resultados[["name", "parameters", "mean_train_seconds", "test_accuracy", "epoch_to_80_val"]])

## Curvas de pérdida y accuracy

Comparamos el progreso en entrenamiento y validación.

In [ ]:
for modelo in ["lenet", "lenet_bn", "vgg11", "vgg11_bn"]:
    display(MostrarImagen(filename=f"results/figures/{modelo}_scratch_seed42_curves.png"))

## ¿BN ayuda a llegar antes al 80%?

Medimos la primera época que alcanza 80% de accuracy en validación.
Un valor vacío indica que no se llegó. El tiempo por época incluye el entrenamiento
y la lectura de datos, pero no la validación ni el guardado.

In [ ]:
display(resultados[["name", "best_val_accuracy", "epoch_to_80_val", "mean_train_seconds", "status"]])
display(MostrarImagen(filename="results/figures/bn_lenet_scratch_seed42.png"))
display(MostrarImagen(filename="results/figures/bn_vgg11_scratch_seed42.png"))

## Probar una LR más alta

Repetimos los cuatro modelos con LR 0.1. Los demás ajustes siguen iguales.
`unstable` indica que hubo valores NaN o infinitos. También hay que mirar las curvas:
una pérdida que oscila o crece puede indicar problemas aunque no aparezca ese estado.

In [ ]:
resultados_lr_alta = run_suite(DEFAULT_SETTINGS, "high_lr", device="cuda")
comparacion = pd.concat([resultados, resultados_lr_alta], ignore_index=True)
display(comparacion[["architecture", "batch_norm", "lr", "best_val_accuracy", "epoch_to_80_val", "status", "failed_epoch"]])
comparacion.to_csv("results/tables/bn_lr_comparison.csv", index=False)
display(export_results("results"))

Las curvas de LR alta se guardan en `results/figures/` como
`bn_lenet_high_lr_seed42.png` y `bn_vgg11_high_lr_seed42.png`.
Un modelo que falla antes de terminar la primera época no tiene curva.

## Comentarios sobre BN

[Ioffe y Szegedy (2015)](https://proceedings.mlr.press/v37/ioffe15.html) propusieron
BN para reducir cambios en las entradas de las capas internas (*Internal Covariate Shift*)
y facilitar el entrenamiento con tasas mayores. BN normaliza por lote y aprende
una escala y un desplazamiento. Nuestras curvas muestran cómo entrenan los modelos,
pero no miden directamente ese cambio de distribución.

- ¿Qué modelo llegó antes al 80%? ¿Alguno no llegó?
- ¿BN mejoró la validación? ¿Cuánto cambió el tiempo por época?
- ¿Qué pasó con LR 0.1? ¿BN ayudó en LeNet y VGG?
- ¿Se separaron mucho los resultados de entrenamiento y validación?

**Respuesta:** [completar con las tablas y curvas]. Una sola semilla no permite
saber cuánto variarían los resultados entre repeticiones.

Continúa con `03_transfer.ipynb`.